# Homework 3 · Beat the AutoML (Python)

**Data Mining 2026/27 — Python part** · doc. dr. Uroš Godnov · **individual homework · 10 points, like Homeworks 1 and 2**

**The case.** A Taiwanese bank wants to know, one month ahead, which credit-card clients will **default** on their next payment, so that it can review their credit limit in time (data: Yeh & Lien, 2009 — 30,000 clients, UCI, CC BY 4.0).

**You know these data from Homework 1**, where you modelled them in AI Studio. Now you solve the same problem in Python, with an honest, leak-proof pipeline, compete with AutoML — and, if you like, with your own Homework 1 model.

**Your job:** build the best *honest* model you can and turn it into a decision. Your benchmark is **AutoGluon** with a 5-minute budget. Beating it is welcome but **not required**. What earns points is a sound process, honest evaluation and a clear business recommendation (see `GRADING.md`).

**Business economics (use these numbers):** flagging a client who *would* default saves on average **€1,000**; flagging a client who would *not* default costs **€200** (lost goodwill and margin). Clients you do not flag change nothing.

**Deliverable:** this notebook, completed, shared as a Colab link (*Anyone with the link → Viewer*). It must run top to bottom with **Runtime → Restart session and run all**.

**Rules**
- **Data:** load it only from the URL in the setup cell; no manual downloads or uploads.
- **Honest evaluation:** the test set is created once (cell provided) and used **once**, at the very end (section 6). Choose models and hyperparameters with cross-validation or a validation split on the training data only.
- **AI use (authorised):** you may use AI tools (Gemini, ChatGPT, Claude, …) to help you code, debug and edit your text. **You must declare** which tools you used and for what in section 8, and you must be able to explain every line of your notebook; a short oral check takes place at the project presentations. Undeclared or unexplainable AI-generated work is not your own work, which the UL SEB exam rules treat as a violation (*Izpitni pravilnik UL EF*, art. 36).
- **Collaboration:** discussing ideas is fine; code and text are written individually.
- **Deadline:** **Sunday 17 January 2027, 23:59**, on Canvas (Homework 3). Late work is accepted only by prior agreement, for a justified reason.

| Section | What you deliver |
|---|---|
| 1 Business understanding | the problem, the decision, the costs in your own words |
| 2 Data understanding | EDA + statistics with conclusions |
| 3 Data preparation | data-quality decisions **log**, leakage check, engineered features |
| 4 Modelling | dummy baseline, logistic regression, a decision tree, a tuned XGBoost — in pipelines, compared by CV |
| 5 AutoML baseline | AutoGluon (≤ 5 min) on the same training data |
| 6 Evaluation | test set once · comparison table · profit-maximising threshold · lift |
| 7 Memo | ≤ 250 words to the head of credit risk |
| 8 AI declaration & reflection | tools, uses, what you verified, what you learned |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/beat-the-automl.ipynb)

## 0 · Setup (provided — do not change)

In [ ]:
import sys

_pandas_was_loaded = "pandas" in sys.modules
%pip install -q autogluon.tabular
if _pandas_was_loaded:
    print("⚠️ Restart the session (Runtime → Restart session) and run from the top.")
else:
    print("✔ AutoGluon installed.")

In [ ]:
import io
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
SEED = 42

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

In [ ]:
CREDIT_URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
archive = zipfile.ZipFile(io.BytesIO(requests.get(CREDIT_URL, timeout=120).content))
credit = pd.read_excel(archive.open("default of credit card clients.xls"), header=1)   # row 0 holds codes X1…Y
credit = credit.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})
print(credit.shape)
credit.head()

In [ ]:
print("✔ data loaded" if credit.shape == (30000, 25) else "⚠️ unexpected shape — rerun the setup cells")

In [ ]:
# The ONE train/test split of this assignment (provided — do not change)
train, test = train_test_split(credit, test_size=0.2, stratify=credit["default"], random_state=SEED)
print("train:", train.shape, "| test:", test.shape, "| default rate:", round(train["default"].mean(), 3))

## 1 · Business understanding

**✍️ Your answer** — In 4–6 sentences: what decision does the bank want to support, what is a false positive and a false negative here, and what is the **break-even probability** for flagging a client (show the calculation from the costs above)?

> *(double-click this cell and write your answer here)*

## 2 · Data understanding — EDA and statistics (on `train` only)

Columns: `LIMIT_BAL` (credit limit, NT$) · `SEX` (1 = male, 2 = female) · `EDUCATION` (1 = graduate school, 2 = university, 3 = high school, 4 = others) · `MARRIAGE` (1 = married, 2 = single, 3 = others) · `AGE` · `PAY_1…PAY_6` (repayment status in the last six months: −1 = paid duly, 1 = one month late, …, 9 = nine months or more late) · `BILL_AMT1…6` (bill statements) · `PAY_AMT1…6` (amounts paid) · target `default`.

In [ ]:
# ✏️ TODO: Explore the training data: quick profile, target rate, distributions of key variables,
#    default rate by EDUCATION, MARRIAGE and PAY_1, and at least one χ² test with Cramér's V and one
#    Mann–Whitney test. Do any codes appear that the documentation above does NOT mention?
# Your EDA here (add as many cells as you need)


**✍️ Your answer** — Your three most important EDA findings (with effect sizes) and the data-quality problems you found.

> *(double-click this cell and write your answer here)*

## 3 · Data preparation

Document **every** decision in the log below (what, why, and its effect). Stateless steps (formulas within a row) may be applied to train and test alike; anything that *learns* from data belongs in a pipeline.

In [ ]:
# ✏️ TODO: Write a function `prepare(df)` that returns a cleaned copy with engineered features
#    (e.g. merge undocumented codes, utilisation = bill / limit, number of late months, payment ratios),
#    and drop columns that must not be features (think: identifiers, leakage). Apply it to train and test.
# def prepare(df):
#     df = df.copy()
#     ...
#     return df
# train_p, test_p = prepare(train), prepare(test)
# X_train, y_train = ...
# X_test, y_test = ...


In [ ]:
#@title ✅ Check — Prepared data { display-mode: "form" }
check('Prepared data', lambda: "ID" not in X_train.columns and "default" not in X_train.columns and len(X_train) == 24000, hint='Drop the identifier and separate the target from the features.')

**✍️ Your answer** — **Decision log** — a table: decision · why · effect. Include your leakage check (is any column known only after the outcome?).

> *(double-click this cell and write your answer here)*

## 4 · Modelling (cross-validation on the training data only)

In [ ]:
# ✏️ TODO: Compare at least: a dummy baseline, logistic regression (scaled numeric + one-hot for SEX, EDUCATION,
#    MARRIAGE), a tuned decision tree and a tuned XGBoost (early stopping on a validation split of the
#    TRAINING data). Store your best XGBoost in `best_model` and collect the CV/validation ROC-AUCs in a
#    DataFrame `cv_results`.
# Your modelling here — pipelines, cross-validation, tuning


## 5 · AutoML baseline (AutoGluon, same training data, ≤ 5 minutes)

In [ ]:
# ✏️ TODO: Train AutoGluon on the prepared TRAINING data (label "default", eval_metric="roc_auc",
#    time_limit ≤ 300). Store the predictor in `automl` and show its leaderboard on its own validation
#    data — NOT on the test set yet. (A warning that CatBoost failed with an ImportError is expected:
#    CatBoost is not installed in Colab, so AutoGluon skips it.)
# from autogluon.tabular import TabularPredictor
# automl = ...


## 6 · Evaluation on the test set (once!)

In [ ]:
# ✏️ TODO: (a) Compute the test ROC-AUC of your best model and of AutoGluon → DataFrame `test_results`.
#    (b) Using the business economics, find the threshold that maximises expected savings on the test set for
#    your best model → `best_threshold`, `best_savings` (compare with threshold 0.5).
#    (c) Show a lift table (top 10/20/30 % of clients by risk).
# p_best = ...
# test_results = ...
# best_threshold, best_savings = ...


## 7 · Memo to the head of credit risk (≤ 250 words)

**✍️ Your answer** — Your recommendation: which model, which threshold, expected savings, limitations and next steps.

> *(double-click this cell and write your answer here)*

## 8 · AI declaration and reflection

**✍️ Your answer** — Which AI tools did you use, for which parts, and how did you check their output? What did you learn, and what would you do differently? Compared with your AI Studio model in Homework 1: what could you do in Python that you could not do there, or the other way round? (Writing "none" for AI tools is acceptable if true.)

> *(double-click this cell and write your answer here)*

In [ ]:
progress()